# catboost_lgbm_timeaware_fusion_v2.ipynb
时间感知CV（按 `issue_time_days` 旧→新前推）+ 稀有合并 + 丢弃流水比率 + **LightGBM 融合**。

In [10]:

# ========= 配置 =========
REF_DATE_STR = "2025-08-31"

TRAIN_CSV = "train.csv"
TESTAA_CSV = "testaa.csv"
TESTAB_CSV = "testab.csv"

# 首选 v2p（已去除 *ratio* 的流水特征）
TRAIN_STM = "train_statement_feature_v2p.csv"
TESTAA_STM = "testaa_statement_feature_v2p.csv"
TESTAB_STM = "testab_statement_feature_v2p.csv"

OUT_DIR = "outputs_fusion_v3"
N_FOLDS = 5
RARE_THRESHOLD = 10
RARE_COLS = ['zip_code','title']
DROP_RATIO_COLS = True

USE_GPU_CAT = True
USE_GPU_LGB = False   # 如环境支持可改 True
RANDOM_STATE = 1337


In [11]:

# CatBoost 固定参数（可按你 v1 调整）
CAT_PARAMS = dict(
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=9.0,
    bootstrap_type='Bernoulli',
    subsample=0.75,
    random_strength=1.5,
    one_hot_max_size=10,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=200
)
# LightGBM 稳健参数（兼容多版本；早停通过 callbacks）
LGBM_PARAMS = dict(
    objective='binary',
    metric='auc',
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_data_in_leaf=60,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l1=0.0,
    lambda_l2=5.0,
    verbosity=-1,
    n_estimators=5000,
    force_col_wise=True,
    n_jobs=-1
    # 如需 GPU: 可添加 "device_type": "gpu" 或 "device": "gpu"
)


In [12]:

import os, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb

os.makedirs(OUT_DIR, exist_ok=True)
REF_DATE = pd.Timestamp(REF_DATE_STR)

def _to_days_since_now(unix_series):
    dt = pd.to_datetime(unix_series, unit='s', utc=True, errors='coerce').dt.tz_convert(None)
    return (REF_DATE - dt).dt.days

def prepare_main_table(df):
    use_cols = [
        'id','title','career','zip_code','residence','loan','term','interest_rate',
        'issue_time','syndicated','installment','record_time','history_time',
        'total_accounts','balance_accounts','balance_limit','balance','level'
    ] + (['label'] if 'label' in df.columns else [])
    df = df[use_cols].copy()
    # 时间类 -> 距参考日的天数
    for c in ['issue_time','record_time','history_time']:
        df[f'{c}_days'] = _to_days_since_now(df[c])
    # 差分
    df['diff_issue_record_days']   = df['issue_time_days'] - df['record_time_days']
    df['diff_issue_history_days']  = df['issue_time_days'] - df['history_time_days']
    df['diff_record_history_days'] = df['record_time_days'] - df['history_time_days']
    # 稳健比例/利用率
    df['utilization']    = (df['balance'] / df['balance_limit']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 10)
    df['accounts_ratio'] = (df['balance_accounts'] / df['total_accounts']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)
    # level 拆分
    def split_level(x):
        if isinstance(x, str) and len(x) >= 2: return x[0], x[1:]
        return 'NA', 'NA'
    lv = df['level'].fillna('NA')
    df['grade'], df['subgrade'] = zip(*lv.map(split_level))
    # 类别转字符串（供 CatBoost 使用）
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna('NA')
    return df

def merge_statement_feats(main_df, stm_path):
    if not os.path.exists(stm_path):
        main_df['has_stm'] = 0
        return main_df
    stm = pd.read_csv(stm_path)
    if 'label' in stm.columns: stm = stm.drop(columns=['label'])
    out = main_df.merge(stm, on='id', how='left')
    stm_cols = [c for c in stm.columns if c!='id']
    out['has_stm'] = (out[stm_cols].notna().any(axis=1)).astype(int)
    out[stm_cols] = out[stm_cols].fillna(0)
    if DROP_RATIO_COLS:
        ratio_cols = [c for c in stm_cols if 'ratio' in c.lower()]
        out = out.drop(columns=ratio_cols, errors='ignore')
    return out

def rare_merge_and_add_freq(train_df, test_dfs, cols, thr=10):
    n = len(train_df)
    for c in cols:
        tr_s = train_df[c].astype(str)
        vc = tr_s.value_counts()
        rare_vals = set(vc[vc < thr].index.tolist())
        # 合并稀有类别
        train_df[c] = tr_s.where(~tr_s.isin(rare_vals), 'OTHER')
        for te in test_dfs:
            if te is None or c not in te.columns: continue
            te_c = te[c].astype(str)
            te[c] = te_c.where(~te_c.isin(rare_vals), 'OTHER')
        # 频次特征（train-only）
        vc2 = train_df[c].value_counts()
        train_df[c + "_freq"] = train_df[c].map(vc2).astype(float).fillna(1.0) / max(n,1)
        for te in test_dfs:
            if te is None or c not in te.columns: continue
            te[c + "_freq"] = te[c].map(vc2).astype(float).fillna(1.0) / max(n,1)
    return train_df, test_dfs

def make_time_series_folds(df, n_folds=5, time_col='issue_time_days'):
    # 旧 -> 新 的累积式前推验证
    order = np.argsort(df[time_col].values)[::-1]
    fold_sizes = np.full(n_folds, len(order)//n_folds, dtype=int)
    fold_sizes[:len(order)%n_folds] += 1
    idx_slices, start = [], 0
    for fs in fold_sizes:
        idx_slices.append(order[start:start+fs])
        start += fs
    folds = []
    for i in range(1, n_folds):
        tr_idx = np.concatenate(idx_slices[:i])
        va_idx = idx_slices[i]
        folds.append((tr_idx, va_idx))
    return folds

def sanitize_cat_params(params, use_gpu=False, y=None, seed=1337):
    p = dict(params)
    if use_gpu and 'rsm' in p:
        p.pop('rsm', None)  # GPU 不支持 rsm（仅 pairwise）
    if p.get('bootstrap_type','').lower()=='bayesian' and 'subsample' in p:
        p.pop('subsample', None); p.setdefault('bagging_temperature', 1.0)
    if use_gpu: p['task_type'] = 'GPU'
    if y is not None:
        pos, neg = int(np.sum(y)), int(len(y)-np.sum(y))
        p['scale_pos_weight'] = float(neg/max(pos,1))
    p.setdefault('loss_function','Logloss'); p.setdefault('eval_metric','AUC'); p.setdefault('verbose', False)
    p['random_seed'] = seed
    return p


In [13]:

def train_catboost_timeaware(df_train, features, cat_cols, params, n_folds=5, seed=1337):
    X = df_train[features].copy()
    y = df_train['label'].astype(int).values
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    p = sanitize_cat_params(params, use_gpu=USE_GPU_CAT, y=y, seed=seed)
    folds = make_time_series_folds(df_train, n_folds=n_folds, time_col='issue_time_days')
    oof = np.zeros(len(y), dtype=float)
    fold_scores, models = [], []
    for f,(tr_idx,va_idx) in enumerate(folds, 1):
        train_pool = Pool(X.iloc[tr_idx], label=y[tr_idx], cat_features=cat_idx)
        valid_pool = Pool(X.iloc[va_idx], label=y[va_idx], cat_features=cat_idx)
        model = CatBoostClassifier(**p)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=300)
        pred = model.predict_proba(valid_pool)[:,1]
        oof[va_idx] = pred
        fold_scores.append(roc_auc_score(y[va_idx], pred))
        models.append(model)
    mean_auc = roc_auc_score(y, oof)
    return mean_auc, fold_scores, oof, models

def to_lgb_categorical(df, cat_cols):
    df2 = df.copy()
    for c in cat_cols:
        if c in df2.columns:
            df2[c] = df2[c].astype('category')
    return df2

def train_lgbm_timeaware(df_train, features, cat_cols, params, n_folds=5, seed=1337):
    X = to_lgb_categorical(df_train[features], cat_cols)
    y = df_train['label'].astype(int).values
    folds = make_time_series_folds(df_train, n_folds=n_folds, time_col='issue_time_days')
    oof = np.zeros(len(y), dtype=float)
    fold_scores, models = [], []
    p = dict(params)
    if USE_GPU_LGB:
        p['device_type'] = 'gpu'; p['device'] = 'gpu'
    for f,(tr_idx,va_idx) in enumerate(folds, 1):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[va_idx], y[va_idx]
        model = lgb.LGBMClassifier(**p, random_state=seed+f)
        callbacks=[lgb.early_stopping(stopping_rounds=300, verbose=False), lgb.log_evaluation(period=0)]
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            categorical_feature=[c for c in cat_cols if c in X_tr.columns],
            callbacks=callbacks
        )
        pred = model.predict_proba(X_va, num_iteration=getattr(model, "best_iteration_", None))[:,1]
        oof[va_idx] = pred
        fold_scores.append(roc_auc_score(y_va, pred))
        models.append(model)
    mean_auc = roc_auc_score(y, oof)
    # 平均增益重要性
    imp = np.zeros(len(features), dtype=float)
    for m in models:
        fi = m.booster_.feature_importance(importance_type='gain')
        cols = m.booster_.feature_name()
        fi_map = {c:v for c,v in zip(cols, fi)}
        imp += np.array([fi_map.get(c, 0.0) for c in features])
    imp /= max(len(models),1)
    fi_df = pd.DataFrame({'feature': features, 'importance_gain': imp}).sort_values('importance_gain', ascending=False)
    return mean_auc, fold_scores, oof, models, fi_df


In [14]:

# ========= 读取与特征合并 =========
tr = pd.read_csv(TRAIN_CSV)
te_aa = pd.read_csv(TESTAA_CSV) if os.path.exists(TESTAA_CSV) else None
te_ab = pd.read_csv(TESTAB_CSV) if os.path.exists(TESTAB_CSV) else None

tr = prepare_main_table(tr)
if te_aa is not None: te_aa = prepare_main_table(te_aa)
if te_ab is not None: te_ab = prepare_main_table(te_ab)

tr = merge_statement_feats(tr, TRAIN_STM)
if te_aa is not None: te_aa = merge_statement_feats(te_aa, TESTAA_STM)
if te_ab is not None: te_ab = merge_statement_feats(te_ab, TESTAB_STM)

# 稀有合并 + 频次特征（仅 train 统计）
tr, [te_aa, te_ab] = rare_merge_and_add_freq(tr, [te_aa, te_ab], cols=RARE_COLS, thr=RARE_THRESHOLD)

drop_cols = ['id','label']
features = [c for c in tr.columns if c not in drop_cols]
cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
cat_cols = [c for c in cat_cols if c in features]

print("Train shape:", tr.shape, "| #features:", len(features), "| #cat:", len(cat_cols))


Train shape: (53480, 47) | #features: 45 | #cat: 10


In [15]:

# ========= 训练：CatBoost =========
mean_auc_cat, fold_cat, oof_cat, models_cat = train_catboost_timeaware(tr, features, cat_cols, CAT_PARAMS, n_folds=N_FOLDS, seed=RANDOM_STATE)
pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_cat': oof_cat}).to_csv(os.path.join(OUT_DIR, "oof_timeaware_cat.csv"), index=False, encoding='utf-8')
print("[CatBoost] OOF AUC =", round(float(mean_auc_cat), 6), "| folds:", [round(float(x),6) for x in fold_cat])


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5899897	best: 0.5899897 (0)	total: 44ms	remaining: 3m 39s
200:	test: 0.6049739	best: 0.6084082 (3)	total: 7.6s	remaining: 3m 1s
bestTest = 0.6084082127
bestIteration = 3
Shrink model to first 4 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5887396	best: 0.5887396 (0)	total: 82.9ms	remaining: 6m 54s
200:	test: 0.6380838	best: 0.6381564 (195)	total: 15s	remaining: 5m 57s
400:	test: 0.6368598	best: 0.6382467 (221)	total: 29.6s	remaining: 5m 39s
bestTest = 0.6382466555
bestIteration = 221
Shrink model to first 222 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6084281	best: 0.6084281 (0)	total: 90ms	remaining: 7m 29s
200:	test: 0.6368266	best: 0.6373654 (149)	total: 16.5s	remaining: 6m 34s
400:	test: 0.6390356	best: 0.6392452 (318)	total: 33.1s	remaining: 6m 20s
600:	test: 0.6381883	best: 0.6392630 (403)	total: 49.9s	remaining: 6m 5s
bestTest = 0.6392629743
bestIteration = 403
Shrink model to first 404 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6241697	best: 0.6241697 (0)	total: 97.7ms	remaining: 8m 8s
200:	test: 0.6692783	best: 0.6693385 (192)	total: 16.1s	remaining: 6m 25s
400:	test: 0.6683132	best: 0.6709344 (364)	total: 32.2s	remaining: 6m 9s
600:	test: 0.6687953	best: 0.6709344 (364)	total: 48.5s	remaining: 5m 54s
bestTest = 0.6709343791
bestIteration = 364
Shrink model to first 365 iterations.
[CatBoost] OOF AUC = 0.602844 | folds: [0.608408, 0.638247, 0.639263, 0.670934]


In [16]:

# ========= 训练：LightGBM =========
mean_auc_lgb, fold_lgb, oof_lgb, models_lgb, fi_lgb = train_lgbm_timeaware(tr, features, cat_cols, LGBM_PARAMS, n_folds=N_FOLDS, seed=RANDOM_STATE+999)
pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_lgb': oof_lgb}).to_csv(os.path.join(OUT_DIR, "oof_timeaware_lgb.csv"), index=False, encoding='utf-8')
fi_lgb.to_csv(os.path.join(OUT_DIR, "feature_importance_lgb.csv"), index=False, encoding='utf-8')
print("[LightGBM] OOF AUC =", round(float(mean_auc_lgb), 6), "| folds:", [round(float(x),6) for x in fold_lgb])
fi_lgb.head(20)


[LightGBM] OOF AUC = 0.598918 | folds: [0.600463, 0.629711, 0.63576, 0.662722]


,feature,importance_gain
2,zip_code,5375.225757
6,interest_rate,2471.710038
5,term,2354.305012
16,level,1504.496996
13,balance_accounts,1031.098371
7,issue_time,811.887228
10,record_time,527.248552
14,balance_limit,482.077279
34,avg_income,453.473304
15,balance,446.083162


In [17]:

# ========= OOF 融合：网格搜加权 =========
best_w, best_auc, best_blend = None, -1.0, None
y_true = tr['label'].values.astype(int)
for w in np.linspace(0, 1, 21):
    blend = w * oof_cat + (1 - w) * oof_lgb
    auc = roc_auc_score(y_true, blend)
    if auc > best_auc:
        best_auc, best_w, best_blend = auc, float(w), blend.copy()

pd.DataFrame({'id': tr['id'], 'label': y_true, 'oof_blend': best_blend}).to_csv(os.path.join(OUT_DIR, "oof_timeaware_blend.csv"), index=False, encoding='utf-8')
print(f"[BLEND] best_w={best_w:.2f} -> OOF AUC={best_auc:.6f}")


[BLEND] best_w=0.35 -> OOF AUC=0.607444


In [18]:

# ========= 测试集预测（cat / lgb / blend） =========
def predict_cat(df):
    X = df[features].copy()
    pool = Pool(X, cat_features=[X.columns.get_loc(c) for c in cat_cols])
    preds = np.mean([m.predict_proba(pool)[:,1] for m in models_cat], axis=0)
    return preds

def predict_lgb(df):
    X = df[features].copy()
    for c in cat_cols:
        if c in X.columns: X[c] = X[c].astype('category')
    preds = np.mean([m.predict_proba(X, num_iteration=getattr(m, "best_iteration_", None))[:,1] for m in models_lgb], axis=0)
    return preds

def save_preds(df, tag, w):
    if df is None:
        print(f"[INFO] 无 {tag} 测试集，跳过。"); return
    p_cat = predict_cat(df); p_lgb = predict_lgb(df); p_blend = w*p_cat + (1-w)*p_lgb
    pd.DataFrame({'id': df['id'], 'prob': p_cat}).to_csv(os.path.join(OUT_DIR, f"test_{tag}_pred_cat.csv"), index=False, encoding='utf-8')
    pd.DataFrame({'id': df['id'], 'prob': p_lgb}).to_csv(os.path.join(OUT_DIR, f"test_{tag}_pred_lgb.csv"), index=False, encoding='utf-8')
    pd.DataFrame({'id': df['id'], 'prob': p_blend}).to_csv(os.path.join(OUT_DIR, f"test_{tag}_pred_blend.csv"), index=False, encoding='utf-8')
    print(f"[SAVE] test_{tag}: cat/lgb/blend 输出完成")

best_w = best_w if 'best_w' in globals() and best_w is not None else 0.5
save_preds(te_aa, "aa", best_w)
save_preds(te_ab, "ab", best_w)


[SAVE] test_aa: cat/lgb/blend 输出完成
[SAVE] test_ab: cat/lgb/blend 输出完成
